In [1]:
pip install anthropic litellm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.0/833.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.0/17.0 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 16.2 MB/s eta 0:00:00


In [2]:
import os
from anthropic import Anthropic
from google.colab import userdata
userdata.get('googleapikey')

def generate_message():
    print("Connecting to local LiteLLM proxy...")

    try:
        # 1. Initialize the official Anthropic Client
        # We point the base_url to your local running LiteLLM server (port 8000)
        client = Anthropic(
            base_url="http://0.0.0.0:8000",
            api_key="fake-key-for-sdk-validation"  # The SDK requires a string, but LiteLLM will bypass it
        )

        # 2. Create the message payload using standard Anthropic structure
        print("Sending prompt to Gemini via proxy...")
        response = client.messages.create(
            model="gemini/gemini-2.5-flash",  # Tells LiteLLM exactly which Gemini model to hit
            max_tokens=500,
            messages=[
                {
                    "role": "user",
                    "content": "Write a short, 3-line poem about why debugging code is satisfying."
                }
            ]
        )

        # 3. Print out the response text safely
        print("\n=== Success! Response From Gemini ===")
        print(response.content[0].text)
        print("=====================================\n")

    except Exception as e:
        print(f"\n❌ An error occurred: {e}")
        print("Make sure your LiteLLM proxy terminal is currently running on port 8000!\n")

if __name__ == "__main__":
    generate_message()

Connecting to local LiteLLM proxy...
Sending prompt to Gemini via proxy...

❌ An error occurred: Connection error.
Make sure your LiteLLM proxy terminal is currently running on port 8000!



In [3]:
import os
import subprocess
from google.colab import userdata
import time

print("--- Setting up LiteLLM Proxy in Colab ---")

# 1. Get Google API Key from Colab secrets and set as environment variable
google_api_key = userdata.get('googleapikey')

if google_api_key:
    os.environ['GEMINI_API_KEY'] = google_api_key # LiteLLM expects GEMINI_API_KEY for Gemini
    os.environ['GOOGLE_API_KEY'] = google_api_key # Also set GOOGLE_API_KEY for broader compatibility
    print("Google API key successfully loaded from Colab secrets and set as environment variables.")
else:
    print("Warning: 'googleapikey' not found in Colab secrets. Please add it to avoid authentication errors.")
    print("Skipping LiteLLM proxy startup as API key is missing.")
    # Depending on requirements, you might want to exit or raise an error here.

# 2. Kill any previously running LiteLLM processes on port 8000
# This helps prevent "Address already in use" errors if the cell is run multiple times.
try:
    print("Attempting to kill any existing LiteLLM processes on port 8000...")
    # fuser command finds processes using a given port and -k kills them.
    # Using check=False to avoid raising an error if no process is found.
    subprocess.run(["fuser", "-k", "8000/tcp"], capture_output=True, text=True, check=False)
    print("Existing LiteLLM processes (if any) killed.")
except FileNotFoundError:
    print("fuser command not found. Cannot kill existing processes automatically.")
except Exception as e:
    print(f"Error killing existing processes: {e}")


# 3. Start LiteLLM proxy server in the background
print("Starting LiteLLM proxy server on port 8000...")
# We use subprocess.Popen to run LiteLLM in a non-blocking way.
# The `preexec_fn=os.setsid` helps detach it from the current process group,
# making it more robust against cell termination, though Colab's environment
# can still be aggressive about killing background jobs.
# For a truly persistent server in Colab, consider using `ngrok` or a dedicated VM.

# LiteLLM command to expose all available models, or specify one.
# Omitting --model allows the client (your Anthropic client) to define which model to use.
command = ["litellm", "--port", "8000"]

# Start the process without waiting for it to finish
proc = subprocess.Popen(command, preexec_fn=os.setsid)

print(f"LiteLLM proxy server started as a background process with PID: {proc.pid}")
print("It might take a few seconds for the server to fully start.")
print("You can now run other cells to interact with the proxy server on http://localhost:8000.")
print("\nImportant Note: LiteLLM processes started this way in Colab may be terminated when the cell finishes execution or when the Colab runtime becomes inactive. For long-running services, other solutions (like `ngrok` or dedicated instances) might be more suitable.")

# Optional: Add a small delay to allow the server to boot up
time.sleep(5)
print("Proxy startup sequence completed.")

--- Setting up LiteLLM Proxy in Colab ---
Google API key successfully loaded from Colab secrets and set as environment variables.
Attempting to kill any existing LiteLLM processes on port 8000...
Existing LiteLLM processes (if any) killed.
Starting LiteLLM proxy server on port 8000...
LiteLLM proxy server started as a background process with PID: 4457
It might take a few seconds for the server to fully start.
You can now run other cells to interact with the proxy server on http://localhost:8000.

Important Note: LiteLLM processes started this way in Colab may be terminated when the cell finishes execution or when the Colab runtime becomes inactive. For long-running services, other solutions (like `ngrok` or dedicated instances) might be more suitable.
Proxy startup sequence completed.


In [ ]:
import os
from anthropic import Anthropic

print("Connecting to local LiteLLM proxy...")

try:
    # FIX: Point explicitly to localhost and include the /anthropic routing suffix
    client = Anthropic(
        base_url="http://localhost:8000/anthropic",
        api_key="fake-key-for-sdk-validation"
    )

    print("Sending prompt to Gemini via proxy...")
    response = client.messages.create(
        model="gemini/gemini-2.5-flash",
        max_tokens=500,
        messages=[
            {"role": "user", "content": "Say 'Connection Successful!' if you can read this."}
        ]
    )

    print("\n=== Response ===")
    print(response.content[0].text)
    print("================\n")

except Exception as e:
    print(f"\n❌ Still throwing an error: {e}")
    print("If it's an API key error, restart the proxy server and make sure you export the key first.")

Connecting to local LiteLLM proxy...
Sending prompt to Gemini via proxy...

❌ Still throwing an error: Error code: 401 - {'error': {'message': '{"type":"error","error":{"type":"authentication_error","message":"invalid x-api-key"},"request_id":"req_011CbPivHhRL25RpnUgun4Nk"}', 'type': 'None', 'param': 'None', 'code': '401'}}
If it's an API key error, restart the proxy server and make sure you export the key first.


In [ ]:
from anthropic import Anthropic

try:
    # 1. Point to the standard local port (No /anthropic suffix needed anymore!)
    client = Anthropic(
        base_url="http://localhost:8000",
        api_key="fake-key-for-sdk-validation"
    )

    # 2. Ask for the model name you defined in your config.yaml
    print("Sending request to local proxy...")
    response = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=500,
        messages=[
            {"role": "user", "content": "Say 'Proxy translation is working!'"}
        ]
    )

    print("\n=== Response ===")
    print(response.content[0].text)
    print("================\n")

except Exception as e:
    print(f"\n❌ Error: {e}")

Sending request to local proxy...

=== Response ===
Proxy translation is working!



In [ ]:
#the actual coding starts from here above are setup to reroute the api calls from anthropicsdk to local proxy server which then
#forwards the request to google gemini api and sends the response back to anthropic sdk and then we can print the response here.

In [ ]:
messages = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=1000,
    messages=[{"role": "user", "content": "What is the capital of India?"}]
)

messages = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=1000,
    messages=[{"role": "user", "content": "Write any other information"}]
)

messages.content[0].text

In [4]:
def add_user_message(messages,text):
    user_message={"role": "user","content":text}
    messages.append(user_message)

def add_assistant_message(messages,text):
    assistant_message={"role": "assistant","content":text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=1000,
        messages=messages
    )
    return message.content[0].text

In [ ]:
# to have the conversation going on :
# the below msgs will store my entire conversation history
messages=[]
add_user_message(messages,"What is the capital of India?")
#u can use the below line to print the content of the list messages.
#messages

answer = chat(messages)

add_assistant_message(messages,answer)

add_user_message(messages,"Write any other information")

answer = chat(messages)
answer

'Certainly!\n\nIndia is also the **most populous country in the world**, surpassing China in 2023.'